# nb_ingest_caged — Ingestão CAGED via BigQuery (Fabric · Dados Públicos)

**Fonte:** `basedosdados.br_me_caged.microdados` (Base dos Dados)  
**Escopo:** 15 municípios · 3 clusters (Santos, Osasco, Mauá) · SP  
**Saídas:** `bronze_caged_raw` → `silver_caged`  
**Granularidade:** município × ano × mes × seção CNAE × subclasse  
**Modo:** `save_delta()` com V-Order (Direct Lake)

> **Pré-requisito:** arquivo de credenciais BigQuery em `/lakehouse/default/Files/bd2024-444413-1084f2b9d765.json`  
> (mesmo arquivo usado pelo `nb_ingest_rais_bigquery`)

In [ ]:
%run ./nb_utils_ibge

%pip install google-cloud-bigquery pyarrow db-dtypes --quiet

from google.cloud import bigquery
import pandas as pd
from pyspark.sql.functions import col, trim, lit
from pyspark.sql.types import IntegerType, LongType

In [ ]:
# Municípios e credenciais — mesma lógica do nb_ingest_rais_bigquery
municipios = get_all_municipios()
municipios_sql = ", ".join([f"'{m}'" for m in municipios])

CREDENTIALS_PATH = "/lakehouse/default/Files/bd2024-444413-1084f2b9d765.json"
client = bigquery.Client.from_service_account_json(CREDENTIALS_PATH)

print(f"[OK] {len(municipios)} municípios carregados dos 3 clusters")
print(f"[OK] Cliente BigQuery inicializado")

## 1. Bronze — Ingestão do BigQuery

Agrega no BigQuery (municipality × ano × mes × CNAE) para reduzir volume de transferência.  
Microdados brutos têm 1 linha por vínculo — a agregação aqui é equivalente ao que o RAIS já faz.

In [ ]:
query = f"""
SELECT
    CAST(ano AS INT64)                   AS ano,
    CAST(mes AS INT64)                   AS mes,
    sigla_uf,
    id_municipio,
    cnae_2_secao,
    cnae_2_subclasse,
    SUM(saldo_movimentacao)              AS saldo_movimentacao,
    COUNT(*)                             AS total_movimentacoes,
    ROUND(AVG(salario_mensal), 2)        AS salario_medio
FROM `basedosdados.br_me_caged.microdados_movimentacao`
WHERE sigla_uf       = 'SP'
  AND id_municipio   IN ({municipios_sql})
  AND ano            >= 2020
GROUP BY
    ano, mes, sigla_uf, id_municipio, cnae_2_secao, cnae_2_subclasse
"""

print("[INFO] Executando query BigQuery CAGED — pode levar alguns minutos...")
df_pandas = client.query(query).to_dataframe()
print(f"[OK] BigQuery retornou {len(df_pandas):,} registros")

df_bronze = spark.createDataFrame(df_pandas)
save_delta(df_bronze, "bronze_caged_raw")
print("[OK] bronze_caged_raw gravada")

## 2. Silver — Enriquecimento com Cluster e Nome do Município

In [ ]:
# Mapas auxiliares — mesmo padrão do nb_ingest_rais_bigquery
cluster_rows = [
    (code, cluster)
    for cluster, codes in CLUSTERS.items()
    for code in codes
]
df_cluster  = spark.createDataFrame(cluster_rows, ["id_municipio_int", "cluster"])
df_nomes    = spark.read.csv("Files/municipio_names.csv", header=True, inferSchema=True)

df_silver = (
    spark.table("bronze_caged_raw")
    .withColumn("id_municipio_int", col("id_municipio").cast(IntegerType()))
    .withColumn("ano",              col("ano").cast(IntegerType()))
    .withColumn("mes",              col("mes").cast(IntegerType()))
    .withColumn("cnae_2_secao",     trim(col("cnae_2_secao")))
    .withColumn("cnae_2_subclasse", trim(col("cnae_2_subclasse")))
    # Join 1: cluster
    .join(df_cluster, "id_municipio_int", "left")
    # Join 2: nome do município
    .join(
        df_nomes.withColumnRenamed("id_municipio", "id_mun_csv"),
        col("id_municipio_int") == col("id_mun_csv"),
        "left"
    )
    .select(
        col("id_municipio_int").alias("id_municipio"),
        col("nome").alias("nome_municipio"),
        "cluster",
        "ano",
        "mes",
        "sigla_uf",
        col("cnae_2_secao").alias("secao_cnae"),
        col("cnae_2_subclasse").alias("subclasse_cnae"),
        col("saldo_movimentacao").cast(LongType()),
        "total_movimentacoes",
        "salario_medio"
    )
)

assert df_silver.count() > 0, "[ERRO] silver_caged vazio antes de gravar"
print(f"[OK] silver_caged: {df_silver.count():,} registros")
display(df_silver.limit(10))

In [ ]:
save_delta(df_silver, "silver_caged")
print("[OK] silver_caged gravada")

# Spot check — cobertura temporal e volumetria por cluster
print("\n=== Cobertura por cluster ===")
df_silver.groupBy("cluster").agg(
    {"ano": "min", "ano": "max", "saldo_movimentacao": "sum"}
).show()

print("\n=== Meses disponíveis ===")
df_silver.select("ano", "mes").distinct().orderBy("ano", "mes").show(50)